# Configs

> Hydra configs (dataclasses + generated YAML) with the default parameters of every MultiGrid environment.


In [ ]:
#| default_exp configs.env

In [ ]:
#| hide
# stable_marl/configs/env.py
from nbdev.showdoc import *

In [ ]:
#| export
import os
from dataclasses import dataclass, field
from typing import Any, List, Optional

from hydra.core.config_store import ConfigStore
from omegaconf import MISSING, OmegaConf


In [ ]:
#| export
@dataclass
class MultiGridEnvConfig:
    """
    Arguments shared by every MultiGrid env (forwarded to `MultiGridEnv.__init__`).

    Env configs below inherit these and add their own arguments / defaults.
    Base arguments that an env fixes internally (e.g. `mission_space`, the grid
    size, or `success_termination_mode` for some envs) are intentionally left out,
    since passing them would clash with, or silently override, the env.
    """
    _target_: str = MISSING            # hydra instantiation target
    _convert_: str = "all"             # pass lists / dicts to the env as plain python objects
    agents: int = 1
    agent_view_size: int = 7
    see_through_walls: bool = False
    allow_agent_overlap: bool = True
    obs_mode: str = "ego"              # 'ego', 'allo' or 'global'
    render_mode: Optional[str] = None  # None, 'human' or 'rgb_array'
    screen_size: Optional[int] = 640
    highlight: bool = True
    tile_size: int = 32                # pixels per cell (rendering and 'pov' observation)
    agent_pov: bool = False


In [ ]:
#| export
@dataclass
class BlockedUnlockPickupEnvConfig(MultiGridEnvConfig):
    _target_: str = "stable_marl.envs.BlockedUnlockPickupEnv"
    room_size: int = 6
    max_steps: Optional[int] = None    # None: 16 * room_size**2
    joint_reward: bool = True
    failure_termination_mode: str = "all"


@dataclass
class EmptyEnvConfig(MultiGridEnvConfig):
    _target_: str = "stable_marl.envs.EmptyEnv"
    size: int = 8
    agent_start_pos: Optional[List[int]] = field(default_factory=lambda: [1, 1])  # None: random
    agent_start_dir: Optional[int] = 0  # 0: right, 1: down, 2: left, 3: up (None: random)
    max_steps: Optional[int] = None    # None: 4 * size**2
    joint_reward: bool = False
    success_termination_mode: str = "any"
    failure_termination_mode: str = "all"


@dataclass
class LockedHallwayEnvConfig(MultiGridEnvConfig):
    _target_: str = "stable_marl.envs.LockedHallwayEnv"
    num_rooms: int = 6
    room_size: int = 5
    max_hallway_keys: int = 1
    max_keys_per_room: int = 2
    max_steps: Optional[int] = None    # None: 8 * num_rooms * room_size**2
    joint_reward: bool = True
    success_termination_mode: str = "any"
    failure_termination_mode: str = "all"


@dataclass
class PlaygroundEnvConfig(MultiGridEnvConfig):
    _target_: str = "stable_marl.envs.PlaygroundEnv"
    room_size: int = 7
    num_rows: int = 3
    num_cols: int = 3
    max_steps: int = 100
    joint_reward: bool = False
    success_termination_mode: str = "any"
    failure_termination_mode: str = "all"


@dataclass
class RedBlueDoorsEnvConfig(MultiGridEnvConfig):
    _target_: str = "stable_marl.envs.RedBlueDoorsEnv"
    size: int = 8                      # grid is (2 * size) x size
    max_steps: Optional[int] = None    # None: 20 * size**2
    joint_reward: bool = True
    success_termination_mode: str = "any"
    failure_termination_mode: str = "any"


@dataclass
class FindGoalEnvConfig(MultiGridEnvConfig):
    _target_: str = "stable_marl.envs.FindGoalEnv"
    size: int = 15
    width: int = 30
    height: int = 20
    num_obstacles: int = 25
    n_clutter: int = 10
    randomize_goal: bool = True
    min_goal_spawn_distance: int = 5
    max_steps: int = 250
    joint_reward: bool = False
    success_termination_mode: str = "all"
    failure_termination_mode: str = "all"
    allow_agent_overlap: bool = False
    see_through_walls: bool = True


@dataclass
class PettingZooWrapperConfig:
    _target_: str = "stable_marl.wrappers.external.PettingZooWrapper"
    env: Any = MISSING                 # an env config (see `register_configs`)


@dataclass
class TorchRLWrapperConfig:
    _target_: str = "stable_marl.wrappers.external.TorchRLPettingZooWrapper"
    env: Any = MISSING                 # an env config (see `register_configs`)


In [ ]:
#| export
# Config name -> dataclass, per Hydra config group
ENV_CONFIGS = {
    'blockedunlockpickup': BlockedUnlockPickupEnvConfig,
    'empty': EmptyEnvConfig,
    'locked_hallway': LockedHallwayEnvConfig,
    'playground': PlaygroundEnvConfig,
    'redbluedoors': RedBlueDoorsEnvConfig,
    'findgoal': FindGoalEnvConfig,
}
WRAPPER_CONFIGS = {
    'pettingzoo': PettingZooWrapperConfig,
    'torchrl': TorchRLWrapperConfig,
}

# Directory of the generated YAML configs (use as `pkg://stable_marl.configs` in a hydra searchpath)
CONFIG_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else None


def register_configs(cs: ConfigStore | None = None):
    """
    Register the env configs (group 'env') and wrapper configs (group 'wrapper')
    in Hydra's ConfigStore, so they can be used in a `defaults` list, e.g.
    `- env: findgoal`.
    """
    cs = cs or ConfigStore.instance()
    for name, node in ENV_CONFIGS.items():
        cs.store(group='env', name=name, node=node)
    for name, node in WRAPPER_CONFIGS.items():
        cs.store(group='wrapper', name=name, node=node)


def write_yaml_configs(config_dir: str):
    """
    (Re)generate `<config_dir>/env/*.yaml` and `<config_dir>/wrapper/*.yaml`
    from the dataclasses, so the YAML files always match them.
    """
    for group, configs in (('env', ENV_CONFIGS), ('wrapper', WRAPPER_CONFIGS)):
        os.makedirs(os.path.join(config_dir, group), exist_ok=True)
        for name, node in configs.items():
            header = (f"# Default config of {node._target_}\n"
                      f"# Generated from stable_marl.configs.env.{node.__name__} "
                      f"(nbs/04_configs.env.ipynb) -- edit the dataclass, not this file.\n")
            with open(os.path.join(config_dir, group, f'{name}.yaml'), 'w') as f:
                f.write(header + OmegaConf.to_yaml(OmegaConf.structured(node)))


## Usage

Every env has a config with its default parameters, available both as a dataclass
(Hydra ConfigStore) and as a YAML file shipped in the package (`stable_marl/configs/env/*.yaml`).
Env configs: `blockedunlockpickup`, `empty`, `locked_hallway`, `playground`, `redbluedoors`, `findgoal`.
Wrapper configs: `pettingzoo`, `torchrl`.

**Option A: YAML files**, by adding the package to Hydra's search path in your primary config:

```yaml
# conf/config.yaml
hydra:
  searchpath:
    - pkg://stable_marl.configs

defaults:
  - env: findgoal
  - _self_

env:            # override any default
  agents: 2
  obs_mode: allo
```

**Option B: dataclasses**, by registering them in the ConfigStore before `@hydra.main` runs:

```python
from stable_marl.configs.env import register_configs
register_configs()

@hydra.main(config_path="conf", config_name="config", version_base=None)
def main(cfg):
    env = hydra.utils.instantiate(cfg.env)   # or stable_marl.utils.env_factory.make_env(cfg.env)
```

**Wrapped envs**: add `- wrapper: pettingzoo` to the defaults and set `wrapper: {env: ${env}}`,
then `instantiate(cfg.wrapper)` builds the env and wraps it.

Overrides work from the command line as usual, e.g. `python train.py env=redbluedoors env.agents=3 env.tile_size=16`.


In [ ]:
#| hide
# Regenerate the YAML configs from the dataclasses
write_yaml_configs('../stable_marl/configs')


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()